# Entraînement du modèle — Score de risque crédit

Dataset : Loan Prediction (Kaggle) — `../data/raw/train.csv`

Étapes (suivant le document de proposition de stage) :
1. Chargement des données
2. Nettoyage (valeurs manquantes)
3. Encodage des variables catégorielles
4. Séparation train / test
5. Entraînement de plusieurs modèles
6. Évaluation des performances
7. Sauvegarde du meilleur modèle

## 1. Chargement des données

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/train.csv')
print(df.shape)
df.head()

(614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [3]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

## 2. Nettoyage des données

On retire `Loan_ID` (identifiant, pas une feature), puis on impute les valeurs manquantes :
- variables catégorielles → mode (valeur la plus fréquente)
- variables numériques → médiane (moins sensible aux valeurs extrêmes que la moyenne)

In [4]:
df = df.drop(columns=['Loan_ID'])

categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
numeric_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

df.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

## 3. Encodage des variables catégorielles

`Dependents` contient la valeur `'3+'` qu'on normalise en `3` avant conversion numérique.
Les autres colonnes catégorielles (y compris la cible `Loan_Status`) sont encodées avec `LabelEncoder`.

In [5]:
from sklearn.preprocessing import LabelEncoder

df['Dependents'] = df['Dependents'].replace('3+', '3').astype(int)

encoders = {}
for col in ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,0,0,0,0,5849,0.0,128.0,360.0,1.0,2,1
1,1,1,1,0,0,4583,1508.0,128.0,360.0,1.0,0,0
2,1,1,0,0,1,3000,0.0,66.0,360.0,1.0,2,1
3,1,1,0,1,0,2583,2358.0,120.0,360.0,1.0,2,1
4,1,0,0,0,0,6000,0.0,141.0,360.0,1.0,2,1


## 4. Séparation en entraînement et test

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((491, 11), (123, 11))

## 5. Entraînement de plusieurs modèles

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)

print('Modèles entraînés.')

C:\anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Modèles entraînés.


In [8]:
!pip install xgboost

## 6. Évaluation des performances

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    results.append({
        'Modèle': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values('F1-score', ascending=False)
results_df

,Modèle,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.861789,0.840000,0.988235,0.908108
2,Random Forest,0.837398,0.849462,0.929412,0.887640
3,XGBoost,0.829268,0.872093,0.882353,0.877193
1,Decision Tree,0.764228,0.825581,0.835294,0.830409


## 7. Sélection et sauvegarde du meilleur modèle

On sélectionne automatiquement le modèle avec le meilleur F1-score, et on sauvegarde
le modèle + les encodeurs + la liste des colonnes attendues (nécessaires pour le
prétraitement identique côté service FastAPI).

In [10]:
import joblib
import os

best_model_name = results_df.iloc[0]['Modèle']
best_model = models[best_model_name]
print(f'Meilleur modèle : {best_model_name}')

os.makedirs('../models', exist_ok=True)

joblib.dump(best_model, '../models/model.pkl')
joblib.dump(encoders, '../models/encoders.pkl')
joblib.dump(list(X.columns), '../models/feature_columns.pkl')

print('Modèle et artefacts sauvegardés dans ml-service/models/')

Meilleur modèle : Logistic Regression
Modèle et artefacts sauvegardés dans ml-service/models/
